### Gold Layer Performance Benchmark (Liquid Clustering)

## Objective
This notebook benchmarks Gold fact table query performance **before and after** applying **Liquid Clustering**.

#### Why do we create duplicate tables?
The Gold fact tables (`coffee.gold.fact_transactions`, `coffee.gold.fact_transaction_items`) are created using DLT streaming tables.
DLT streaming tables **do not allow OPTIMIZE**, so we create duplicate "benchmark" copies as normal Delta tables.

#### Tables Created
- `coffee.gold.fact_transactions_lc`
- `coffee.gold.fact_transaction_items_lc`

#### Benchmark Strategy
-  Create duplicate benchmark tables.
-  Apply Liquid Clustering + OPTIMIZE.
-  Run the same 5 queries on clustered benchmark tables.
-  Compare runtime results.



In [0]:
-- =========================================================
-- Create duplicate benchmark tables in coffee.gold
-- =========================================================

-- Drop old benchmark copies if they exist
DROP TABLE IF EXISTS coffee.gold.fact_transactions_lc;
DROP TABLE IF EXISTS coffee.gold.fact_transaction_items_lc;


-- Create duplicates from original DLT streaming facts
CREATE TABLE coffee.gold.fact_transactions_lc
AS
SELECT * FROM coffee.gold.fact_transactions;

CREATE TABLE coffee.gold.fact_transaction_items_lc
AS
SELECT * FROM coffee.gold.fact_transaction_items;


In [0]:
-- =========================================================
-- Apply Liquid Clustering on the duplicate benchmark tables
-- =========================================================

-- Transactions:
-- created_at -> time filtering
-- store_id   -> store analytics
-- user_id    -> customer analytics
ALTER TABLE coffee.gold.fact_transactions_lc
CLUSTER BY (created_at, store_id, user_id);

-- Transaction Items:
-- created_at      -> time filtering
-- transaction_id  -> joins to fact_transactions
-- item_id         -> item analytics
ALTER TABLE coffee.gold.fact_transaction_items_lc
CLUSTER BY (created_at, transaction_id, item_id);


In [0]:
-- =========================================================
-- OPTIMIZE benchmark tables to physically rewrite files
-- =========================================================
OPTIMIZE coffee.gold.fact_transactions_lc;
OPTIMIZE coffee.gold.fact_transaction_items_lc;


In [0]:
-- =========================================================
-- Q1: Monthly Sales Trend
-- =========================================================
SELECT
  date_trunc('month', created_at) AS sales_month,
  ROUND(SUM(final_amount), 2) AS total_sales
FROM coffee.gold.fact_transactions_lc
GROUP BY date_trunc('month', created_at)
ORDER BY sales_month;


In [0]:
-- =========================================================
-- Q2: Store Performance (Last 3 Months)
-- =========================================================
WITH max_dt AS (
  SELECT MAX(created_at) AS max_created_at
  FROM coffee.gold.fact_transactions_lc
)
SELECT
  store_id,
  ROUND(SUM(final_amount), 2) AS total_sales
FROM coffee.gold.fact_transactions_lc
WHERE created_at >= add_months((SELECT max_created_at FROM max_dt), -3)
GROUP BY store_id
ORDER BY total_sales DESC;

In [0]:
-- =========================================================
-- Q3: Top 10 Customers by Spend
-- =========================================================
SELECT
  item_id,
  ROUND(SUM(subtotal), 2) AS total_revenue
FROM coffee.gold.fact_transaction_items_lc
GROUP BY item_id
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
-- =========================================================
-- Q4: Top 10 Menu Items by Revenue
-- =========================================================
SELECT
  item_id,
  ROUND(SUM(subtotal), 2) AS total_revenue
FROM coffee.gold.fact_transaction_items_lc
GROUP BY item_id
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
-- =========================================================
-- Q5: Join Drilldown (Store + Item Revenue)
-- =========================================================
SELECT
  t.store_id,
  i.item_id,
  ROUND(SUM(i.subtotal), 2) AS total_revenue
FROM coffee.gold.fact_transactions_lc t
JOIN coffee.gold.fact_transaction_items_lc i
  ON t.transaction_id = i.transaction_id
GROUP BY t.store_id, i.item_id
ORDER BY total_revenue DESC;

## Benchmark Results (Baseline vs Liquid Clustering)

| Query ID | Query Name | Baseline Runtime (Run 2) | Liquid Clustering Runtime (Run 2) |
|---------|------------|---------------------------|-----------------------------------|
| Q1 | Monthly Sales Trend | 2.23 |1.37  |
| Q2 | Store Performance (Last 3 Months) | 2.47 | 1.56 |
| Q3 | Top 10 Customers by Spend | 1.69 | 1.46 |
| Q4 | Top 10 Menu Items by Revenue | 1.59 | 1.36 |
| Q5 | Join Drilldown (Store + Item Revenue) | 6.19 | 4.68 |
